# XGBoost para depressão (NHIS 2025): com e sem variáveis de ansiedade

Este notebook repete com o **XGBoost** a análise feita com o Random Forest (`../Random Forest/06_random_forest_depressao_2025.ipynb`) e com o KNN: comparar o modelo com e sem variáveis de ansiedade e testar se o resultado depende das variáveis cognitivas e de solidão.

## Hipótese

> É possível prever depressão a partir de dados de saúde física e do contexto de vida, **sem** depender de sintomas de transtornos mentais.

Esperamos desempenho **pior** sem os sintomas mentais. A hipótese se sustenta se o modelo ainda tiver poder preditivo real, bem acima do acaso.

## Cenários

1. **Cenário A (exploratório, com ansiedade):** removemos de $X$ apenas os sintomas de depressão (PHQ) e mantemos as variáveis de ansiedade (GAD-7).
2. **Cenário B (hipótese, sem sintomas mentais):** removemos também as variáveis de ansiedade.

## Roteiro

1. Preparação dos dados e cenários.
2. Função de treino e avaliação.
3. Cenários A e B e comparativo.
4. Ajuste de hiperparâmetros (`RandomizedSearchCV`) no cenário B.
5. Teste de sensibilidade: o resultado depende das variáveis cognitivas e de solidão?
6. Priorizando o recall: rastreio em saúde.
7. O que mais dá para fazer pelo recall no XGBoost.
8. Interpretação e conclusão.

## Desenho do experimento

- Mesma base, **mesmas linhas** (registros sem `-1` nas variáveis do cenário A), mesmo alvo binário (`0` = sem depressão, `1` = com depressão) e mesmo split treino/teste (80/20 estratificado, `random_state=42`) do Random Forest e do KNN. Os resultados dos três modelos são comparáveis. A justificativa do alvo binário está no notebook do Random Forest.
- **O `PESO_AMOSTRAL` não é usado no treino.** O desbalanceamento (cerca de 21% de positivos) é tratado com `scale_pos_weight` (negativos ÷ positivos no treino), o equivalente ao `class_weight='balanced'` do Random Forest. O peso entra apenas no cálculo das métricas de avaliação, e as matrizes de confusão são sempre em contagem de registros.
- Classificamos como "com depressão" quando a probabilidade é maior ou igual a 0,5. Com o `scale_pos_weight`, o modelo prioriza o recall. Por isso a precisão é menor e o recall maior que no Random Forest. O **ROC-AUC** não depende do limiar e serve para comparar os modelos, mas **a prioridade prática é o recall**.
- **Por que priorizar o recall:** em saúde, os dois erros não custam o mesmo. Um falso negativo é uma pessoa com depressão que o modelo não identifica e que, portanto, não é encaminhada para avaliação. Um falso positivo é uma pessoa sem depressão que passa por uma avaliação a mais. Em um rastreio, é preferível encontrar mais casos, mesmo com mais falsos positivos. Por isso usamos `scale_pos_weight` (que favorece a classe com depressão) e, na seção 6, mostramos quanto recall é possível ganhar baixando o limiar e quanto isso custa em falsos positivos. Na seção 7, testamos o que mais dá para fazer pelo recall com o XGBoost.
- **Diferença em relação à versão anterior deste notebook** (ROC-AUC de 0,872): ela usava registros completos só nas 46 variáveis do cenário B, treinava com `PESO_AMOSTRAL` e otimizava o F1. Aqui as linhas, o split e o treino são os mesmos dos outros modelos, então os números não são comparáveis com os da versão anterior.

# 1. Preparação dos dados e definição dos cenários

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

PESO_COLUNA = 'PESO_AMOSTRAL'
TARGET_ORIGINAL = 'SEVERIDADE_DEPRESSAO_PHQ'
TARGET_BIN = 'DEPRESSAO_PHQ_SIMPLIFICADO'
RANDOM_STATE = 42

# Sintomas diretos de depressão (PHQ): removidos nos DOIS cenários
COLUNAS_DEPRESSAO = [
    'FREQ_PERDA_INTERESSE_2SEM', 'FREQ_SENTIU_DEPRIMIDO_2SEM',
    'FREQ_DIFICULDADE_DORMIR_2SEM', 'FREQ_CANSASO_2SEM',
    'FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM', 'FREQ_AUTOAVALIACAO_NEGTIVA_2SEM',
    'FREQ_DIFICULDADE_CONCENTRACAO_2SEM', 'MODIFICACAO_VELOCIDADE_FALA',
    'FREQ_DEPRESSAO', 'REMEDIO_DEPRESSAO', 'HISTORICO_DEPRESSAO',
    'INTENSIDADE_DEPRESSAO', 'SEVERIDADE_DEPRESSAO_PHQ'
]

# Ansiedade (GAD-7): mantida no cenário A, removida no cenário B
COLUNAS_ANSIEDADE = [
    'HISTORICO_TRANSTORNO_ANSIEDADE', 'MED_ANSIEDADE',
    'FREQ_ANSIEDADE', 'INTENSIDADE_ANSIEDADE',
    'FREQ_ANSIEDADE_2SEM', 'FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM',
    'FREQ_PREOCUPACAO_DESNECESSARIA_2SEM', 'FREQ_DIFICULDADE_DESCANSAR_2SEM',
    'FREQ_AGITACAO_INQUIETA', 'FREQ_IRRITABILIDADE_2SEM',
    'FREQ_MEDO_2SEM', 'SEVERIDADE_ANSIEDADE'
]

# Carregar a base limpa
df = pd.read_csv(Path('../../01 Preparação dos dados/dados/limpos') / 'adult25_limpo.csv').set_index('ID_DOMICILIO')

# Garante que nenhuma coluna foi digitada errada (o drop abaixo ignoraria erros silenciosamente)
ausentes = [c for c in COLUNAS_DEPRESSAO + COLUNAS_ANSIEDADE if c not in df.columns]
assert not ausentes, f'Colunas inexistentes na base: {ausentes}'

# Remover registros indefinidos no alvo (-1 e 5) e binarizar: 0 = sem depressão | 1 = com depressão
df = df[~df[TARGET_ORIGINAL].isin([-1, 5])].copy()
df[TARGET_BIN] = (df[TARGET_ORIGINAL] != 1).astype(int)

CENARIOS = {
    'A: com ansiedade (exploratório)': COLUNAS_DEPRESSAO,
    'B: sem sintomas mentais (hipótese)': COLUNAS_DEPRESSAO + COLUNAS_ANSIEDADE,
}
nome_a, nome_b = CENARIOS
BASE_REMOVER = [TARGET_BIN, PESO_COLUNA]

# Mesmas linhas nos dois cenários: registros sem -1 nas variáveis do cenário A (o mais restritivo)
X_cenario_a = df.drop(columns=BASE_REMOVER + CENARIOS[nome_a])
df_modelo = df[(X_cenario_a != -1).all(axis=1).to_numpy()].copy()

# Mesmo split nos dois cenários (por posição).
# Só existem DUAS partições: treino (80%) e teste (20%). Não há conjunto de validação separado:
# a validação é a validação cruzada feita DENTRO do treino (hiperparâmetros e limiar). O teste só mede o resultado.
posicoes = np.arange(len(df_modelo))
pos_tr, pos_te = train_test_split(
    posicoes, test_size=0.2, random_state=RANDOM_STATE, stratify=df_modelo[TARGET_BIN]
)
y_tr, y_te = df_modelo[TARGET_BIN].iloc[pos_tr], df_modelo[TARGET_BIN].iloc[pos_te]
w_te = df_modelo[PESO_COLUNA].iloc[pos_te]
prevalencia = np.average(y_te, weights=w_te)

# Desbalanceamento: peso da classe positiva = negativos / positivos no treino (equivale a class_weight='balanced')
SCALE_POS = (y_tr == 0).sum() / (y_tr == 1).sum()

print(f'Registros no experimento: {len(df_modelo)} (treino: {len(pos_tr)} | teste: {len(pos_te)})')
print(f'Proporção de depressão (ponderada pelo peso amostral): {np.average(df_modelo[TARGET_BIN], weights=df_modelo[PESO_COLUNA]):.1%}')
print(f'scale_pos_weight (negativos / positivos no treino): {SCALE_POS:.2f}')
for nome, remover in CENARIOS.items():
    n_feat = df_modelo.drop(columns=BASE_REMOVER + remover).shape[1]
    print(f'Cenário {nome}: {n_feat} variáveis preditoras')

Registros no experimento: 22084 (treino: 17667 | teste: 4417)
Proporção de depressão (ponderada pelo peso amostral): 21.2%
scale_pos_weight (negativos / positivos no treino): 3.72
Cenário A: com ansiedade (exploratório): 58 variáveis preditoras
Cenário B: sem sintomas mentais (hipótese): 46 variáveis preditoras


# 2. Função de treino e avaliação

A mesma função é usada nos dois cenários, então a **única diferença** entre eles é o conjunto de variáveis. A configuração-base do XGBoost é 300 árvores de profundidade 4, `learning_rate = 0,05`, `subsample = 0,8` e `colsample_bytree = 0,8`. As métricas da classe positiva (com depressão) são ponderadas pelo `PESO_AMOSTRAL`, e a matriz de confusão é em **contagem de registros reais** do teste, sem peso. A importância das variáveis é o ganho médio de cada variável, somado entre as colunas One-Hot da mesma variável original. A função `avaliar` recebe o limiar de decisão (padrão: probabilidade 0,5), o que permite, na seção 6, priorizar o recall.

In [2]:
def preparar_matrizes(colunas_remover):
    """Pré-processamento: padroniza a variável numérica e aplica One-Hot nas categóricas."""
    X = df_modelo.drop(columns=BASE_REMOVER + colunas_remover)
    X_tr, X_te = X.iloc[pos_tr], X.iloc[pos_te]
    num_cols = [c for c in ['RAZAO_RENDA_POBREZA'] if c in X.columns]
    cat_cols = [c for c in X.columns if c not in num_cols]
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols),
    ])
    # Sem vazamento do teste: escalonador e One-Hot aprendem só no treino (fit) e apenas aplicam no teste (transform)
    return X, preprocessor, preprocessor.fit_transform(X_tr), preprocessor.transform(X_te)


def novo_xgb(**parametros):
    base = dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                scale_pos_weight=SCALE_POS, eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1)
    base.update(parametros)
    return XGBClassifier(**base)


def avaliar(y_proba, limiar=0.5):
    y_pred = (y_proba >= limiar).astype(int)
    metricas = {
        'ROC-AUC': roc_auc_score(y_te, y_proba, sample_weight=w_te),
        'Acurácia': accuracy_score(y_te, y_pred, sample_weight=w_te),
        'Precisão (depressão)': precision_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
        'Recall (depressão)': recall_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
        'F1 (depressão)': f1_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
    }
    return metricas, confusion_matrix(y_te, y_pred)


def importancia_agregada(modelo, preprocessor, X):
    """Soma a importância (ganho) das colunas One-Hot de cada variável original."""
    importancias = pd.Series(modelo.feature_importances_, index=preprocessor.get_feature_names_out())
    originais = sorted(X.columns, key=len, reverse=True)
    agregada = {}
    for nome_col, imp in importancias.items():
        base = nome_col.split('__', 1)[1]
        dono = next((c for c in originais if base == c or base.startswith(c + '_')), base)
        agregada[dono] = agregada.get(dono, 0) + imp
    return pd.Series(agregada).sort_values(ascending=False)


def treinar_avaliar(colunas_remover, **parametros):
    X, preprocessor, X_tr_p, X_te_p = preparar_matrizes(colunas_remover)
    modelo = novo_xgb(**parametros).fit(X_tr_p, y_tr)
    metricas, matriz = avaliar(modelo.predict_proba(X_te_p)[:, 1])
    return {'Nº de variáveis': X.shape[1], **metricas}, matriz, importancia_agregada(modelo, preprocessor, X)

resultados = {}

# 3. Cenários A e B

**Cenário A (exploratório):** mantém as variáveis de ansiedade. Mostra o desempenho quando há informação de saúde mental disponível.

In [3]:
resultados[nome_a] = treinar_avaliar(CENARIOS[nome_a])
metricas_a, matriz_a, imp_a = resultados[nome_a]

print(f'--- {nome_a} ---')
for k, v in metricas_a.items():
    print(f'{k}: {v:.4f}' if isinstance(v, float) else f'{k}: {v}')
print('\nMatriz de confusão (nº de registros do teste; linhas = real, colunas = previsto):')
print(matriz_a)
print('\nTop 10 variáveis mais importantes (ganho agregado):')
print(imp_a.head(10).round(4).to_string())

--- A: com ansiedade (exploratório) ---
Nº de variáveis: 58
ROC-AUC: 0.9146
Acurácia: 0.8534
Precisão (depressão): 0.6245
Recall (depressão): 0.8082
F1 (depressão): 0.7045

Matriz de confusão (nº de registros do teste; linhas = real, colunas = previsto):
[[3014  468]
 [ 176  759]]

Top 10 variáveis mais importantes (ganho agregado):
IDADE                                  0.1136
TIPO_DIFICULDADE_COGNITIVA             0.1008
FREQ_ANSIEDADE                         0.0799
HISTORICO_TRANSTORNO_ANSIEDADE         0.0724
SEVERIDADE_ANSIEDADE                   0.0673
FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM    0.0493
FREQ_ANSIEDADE_2SEM                    0.0477
FREQ_PREOCUPACAO_DESNECESSARIA_2SEM    0.0444
FREQ_SOLITARIO                         0.0316
FREQ_DIFICULDADE_DESCANSAR_2SEM        0.0296


**Cenário B (hipótese):** sem sintomas de depressão **nem** de ansiedade. Sobram variáveis sociodemográficas, de saúde física, de acesso a serviços de saúde e de contexto social.

In [4]:
resultados[nome_b] = treinar_avaliar(CENARIOS[nome_b])
metricas_b, matriz_b, imp_b = resultados[nome_b]

print(f'--- {nome_b} ---')
for k, v in metricas_b.items():
    print(f'{k}: {v:.4f}' if isinstance(v, float) else f'{k}: {v}')
print('\nMatriz de confusão (nº de registros do teste; linhas = real, colunas = previsto):')
print(matriz_b)
print('\nTop 10 variáveis mais importantes (ganho agregado):')
print(imp_b.head(10).round(4).to_string())

--- B: sem sintomas mentais (hipótese) ---
Nº de variáveis: 46
ROC-AUC: 0.8536
Acurácia: 0.7988
Precisão (depressão): 0.5249
Recall (depressão): 0.7327
F1 (depressão): 0.6116

Matriz de confusão (nº de registros do teste; linhas = real, colunas = previsto):
[[2807  675]
 [ 242  693]]

Top 10 variáveis mais importantes (ganho agregado):
TIPO_DIFICULDADE_COGNITIVA      0.2206
IDADE                           0.1969
FREQ_SOLITARIO                  0.0804
FREQUENCIA_FALHA_MEMORIA        0.0481
DOR_CRONICA                     0.0393
SUPORTE_SOCIAL                  0.0391
AUTOAVALIACAO_SAUDE_GERAL       0.0375
CATEGORIA_RAZAO_POBREZA         0.0286
ESCOLARIDADE_MAXIMA_FAMILIA     0.0234
LIMITACAO_TRABALHO_POR_SAUDE    0.0212


## Comparativo entre os cenários

A coluna **Variação (B − A)** mostra o que se perde ao retirar a ansiedade. A coluna **Referência (acaso)** mostra o que um modelo sem informação conseguiria: ROC-AUC de 0,5 e precisão igual à prevalência.

In [5]:
tabela = pd.DataFrame({nome: res[0] for nome, res in resultados.items()})
tabela['Variação (B − A)'] = tabela[nome_b] - tabela[nome_a]
tabela['Referência (acaso)'] = [np.nan, 0.5, np.nan, prevalencia, np.nan, np.nan]

print(f'Prevalência de depressão no teste (ponderada): {prevalencia:.1%}\n')
tabela.round(4)

Prevalência de depressão no teste (ponderada): 21.6%



,A: com ansiedade (exploratório),B: sem sintomas mentais (hipótese),Variação (B − A),Referência (acaso)
Nº de variáveis,58.0000,46.0000,-12.0000,NaN
ROC-AUC,0.9146,0.8536,-0.0610,0.5000
Acurácia,0.8534,0.7988,-0.0546,NaN
Precisão (depressão),0.6245,0.5249,-0.0995,0.2163
Recall (depressão),0.8082,0.7327,-0.0755,NaN
F1 (depressão),0.7045,0.6116,-0.0929,NaN


### Resultado (configuração-base)

- O ROC-AUC cai de **0,915** (A) para **0,854** (B), uma perda de 0,061, e o recall de 80,8% para 73,3%.
- No cenário A, 7 das 10 variáveis mais importantes são de ansiedade (`FREQ_ANSIEDADE`, `HISTORICO_TRANSTORNO_ANSIEDADE`, `SEVERIDADE_ANSIEDADE`, entre outras), atrás apenas de idade e dificuldade cognitiva. Isso confirma a comorbidade entre os dois quadros e é o motivo de usar o cenário A só como referência.
- Sem nenhum sintoma de saúde mental, o cenário B fica **0,35 de ROC-AUC acima do acaso**: a precisão é de 52,5%, **2,4 vezes a prevalência** (21,6%). O modelo usa só dados físicos, funcionais e sociais.
- **Sensibilidade (recall = 73,3%):** o modelo encontra **693 dos 935 casos de depressão do teste** (74,1% por contagem; o recall de 73,3% é ponderado pelo `PESO_AMOSTRAL`) e deixa de detectar **242** (26% dos casos reais). O `scale_pos_weight` desloca a fronteira em favor da classe com depressão, o que sustenta esse recall. No cenário A, são 759 casos encontrados e 176 perdidos (19%).
- **Custo em precisão:** o ganho de sensibilidade vem com falsos alarmes. **675 das 3.482 pessoas sem depressão** do teste (19%) são sinalizadas, e cerca de 52 em cada 100 pessoas sinalizadas têm depressão segundo o PHQ. Em um rastreio esse custo é aceitável, e a seção 6 mostra até onde vale aumentar o recall.
- No cenário B, a importância se concentra em `TIPO_DIFICULDADE_COGNITIVA` (0,22) e `IDADE` (0,20), seguidas de solidão, falha de memória, dor crônica, suporte social e autoavaliação de saúde. A seção 5 testa o quanto o resultado depende das variáveis cognitivas e da solidão.

# 4. Ajuste de hiperparâmetros (cenário B)

Buscamos número de árvores, profundidade, taxa de aprendizado, amostragem de linhas e colunas, `min_child_weight` e regularização L2 com `RandomizedSearchCV` (20 combinações, validação cruzada de 3 partes), **só no cenário B**, que testa a hipótese. A seleção usa o ROC-AUC e **só os dados de treino**; o teste não participa da escolha. O `scale_pos_weight` fica fixo. Otimizamos o ROC-AUC, que avalia o modelo em todos os limiares, e não o recall: otimizar só o recall levaria a um modelo que sinaliza todo mundo (recall de 100% e nenhuma utilidade). A prioridade pelo recall é tratada depois, na escolha do limiar (seção 6).

In [6]:
_, _, X_tr_b, X_te_b = preparar_matrizes(CENARIOS[nome_b])

ESPACO_BUSCA = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.03, 0.05, 0.1],
    'subsample': [0.7, 0.85, 1.0],
    'colsample_bytree': [0.5, 0.7, 1.0],
    'min_child_weight': [1, 3, 5],
    'reg_lambda': [1, 5, 10],
}

# A busca otimiza o ROC-AUC (não ponderado) em 3 partes do treino; o teste não participa
busca = RandomizedSearchCV(
    novo_xgb(), ESPACO_BUSCA, n_iter=20, scoring='roc_auc', cv=3,
    random_state=RANDOM_STATE, n_jobs=1,
)
# Hiperparâmetros escolhidos só com validação cruzada no treino (cv=3): o teste não participa da busca
busca.fit(X_tr_b, y_tr)
print('Melhores hiperparâmetros:', busca.best_params_)

modelo_aj = busca.best_estimator_
metricas_aj, matriz_aj = avaliar(modelo_aj.predict_proba(X_te_b)[:, 1])
print('\nMatriz de confusão do modelo ajustado (nº de registros do teste):')
print(matriz_aj)
comparacao = pd.DataFrame({'B base': {k: resultados[nome_b][0][k] for k in metricas_aj}, 'B ajustado': metricas_aj})
comparacao['Variação'] = comparacao['B ajustado'] - comparacao['B base']
comparacao.round(4)

Melhores hiperparâmetros: {'subsample': 0.85, 'reg_lambda': 10, 'n_estimators': 400, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.5}

Matriz de confusão do modelo ajustado (nº de registros do teste):
[[2792  690]
 [ 243  692]]


,B base,B ajustado,Variação
ROC-AUC,0.8536,0.8556,0.0020
Acurácia,0.7988,0.7963,-0.0025
Precisão (depressão),0.5249,0.5207,-0.0043
Recall (depressão),0.7327,0.7324,-0.0003
F1 (depressão),0.6116,0.6086,-0.0030


### Resultado do ajuste

- A busca escolheu árvores mais rasas (`max_depth = 3`), mais árvores (400), `colsample_bytree = 0,5` e regularização L2 maior (`reg_lambda = 10`), ou seja, um modelo **mais simples e mais regularizado**.
- O ganho é pequeno: o ROC-AUC vai de 0,8536 para **0,8556** (+0,002), e precisão, recall e F1 variam menos de 0,005. O limite está na **informação disponível nas variáveis**, não nos hiperparâmetros. Foi o mesmo padrão do Random Forest.

# 5. Teste de sensibilidade: o resultado depende das variáveis cognitivas e de solidão?

`TIPO_DIFICULDADE_COGNITIVA` é a variável mais importante do cenário B e vem da pergunta sobre dificuldade para **lembrar ou se concentrar**. Dificuldade de concentração também é item do PHQ, que já removemos. Há o risco de essas variáveis reintroduzirem sintomas depressivos no modelo. O mesmo cuidado vale para a solidão. Repetimos o cenário B (mesmas linhas, mesmo split e os hiperparâmetros ajustados) retirando progressivamente:

1. `TIPO_DIFICULDADE_COGNITIVA` sozinha;
2. o **bloco cognitivo** inteiro (`TIPO_DIFICULDADE_COGNITIVA`, `QTD_DIFICULDADE_COGNITIVA` e `FREQUENCIA_FALHA_MEMORIA`, que vêm da mesma pergunta);
3. o bloco cognitivo **e** `FREQ_SOLITARIO`.

Retirar só uma variável é um teste fraco, porque as outras do bloco carregam a mesma informação. Por isso o passo 2 é o mais informativo.

In [7]:
BLOCO_COGNITIVO = ['TIPO_DIFICULDADE_COGNITIVA', 'QTD_DIFICULDADE_COGNITIVA', 'FREQUENCIA_FALHA_MEMORIA']
SOLIDAO = ['FREQ_SOLITARIO']
COLUNAS_B = CENARIOS[nome_b]

SENSIBILIDADE = {
    'B (referência)': COLUNAS_B,
    'B sem TIPO_DIFICULDADE_COGNITIVA': COLUNAS_B + BLOCO_COGNITIVO[:1],
    'B sem bloco cognitivo': COLUNAS_B + BLOCO_COGNITIVO,
    'B sem bloco cognitivo e sem solidão': COLUNAS_B + BLOCO_COGNITIVO + SOLIDAO,
}

res_sens = {nome: treinar_avaliar(remover, **busca.best_params_) for nome, remover in SENSIBILIDADE.items()}

tabela_sens = pd.DataFrame({nome: res[0] for nome, res in res_sens.items()}).T
tabela_sens['Variação do ROC-AUC vs. B'] = tabela_sens['ROC-AUC'] - tabela_sens.loc['B (referência)', 'ROC-AUC']
tabela_sens['Nº de variáveis'] = tabela_sens['Nº de variáveis'].astype(int)
print(f'Referência de acaso: ROC-AUC 0,5 e precisão {prevalencia:.1%} (prevalência)\n')
print('Top 5 variáveis de cada cenário:')
for nome, res in res_sens.items():
    print(f'  {nome}: {", ".join(res[2].head(5).index)}')
tabela_sens.round(4)

Referência de acaso: ROC-AUC 0,5 e precisão 21.6% (prevalência)

Top 5 variáveis de cada cenário:
  B (referência): TIPO_DIFICULDADE_COGNITIVA, IDADE, FREQ_SOLITARIO, SUPORTE_SOCIAL, FREQUENCIA_FALHA_MEMORIA
  B sem TIPO_DIFICULDADE_COGNITIVA: FREQUENCIA_FALHA_MEMORIA, IDADE, FREQ_SOLITARIO, SUPORTE_SOCIAL, QTD_DIFICULDADE_COGNITIVA
  B sem bloco cognitivo: FREQ_SOLITARIO, IDADE, SUPORTE_SOCIAL, LIMITACAO_TRABALHO_POR_SAUDE, DOR_CRONICA
  B sem bloco cognitivo e sem solidão: IDADE, SUPORTE_SOCIAL, LIMITACAO_TRABALHO_POR_SAUDE, AUTOAVALIACAO_SAUDE_GERAL, DOR_CRONICA


,Nº de variáveis,ROC-AUC,Acurácia,Precisão (depressão),Recall (depressão),F1 (depressão),Variação do ROC-AUC vs. B
B (referência),46,0.8556,0.7963,0.5207,0.7324,0.6086,0.0000
B sem TIPO_DIFICULDADE_COGNITIVA,45,0.8528,0.7929,0.5150,0.7263,0.6027,-0.0028
B sem bloco cognitivo,43,0.8387,0.7775,0.4904,0.7308,0.5870,-0.0169
B sem bloco cognitivo e sem solidão,42,0.8106,0.7525,0.4543,0.7170,0.5562,-0.0450


### Resultado do teste de sensibilidade

| Cenário | Variáveis | ROC-AUC | Precisão | Recall | F1 |
|---|---|---|---|---|---|
| B (referência) | 46 | 0,856 | 52,1% | 73,2% | 0,609 |
| sem `TIPO_DIFICULDADE_COGNITIVA` | 45 | 0,853 | 51,5% | 72,6% | 0,603 |
| sem bloco cognitivo | 43 | 0,839 | 49,0% | 73,1% | 0,587 |
| sem bloco cognitivo e sem solidão | 42 | **0,811** | 45,4% | 71,7% | 0,556 |

- **Sem o `TIPO`:** o ROC-AUC cai só 0,003, porque `FREQUENCIA_FALHA_MEMORIA` e `QTD_DIFICULDADE_COGNITIVA` assumem o papel dela.
- **Sem o bloco cognitivo:** o ROC-AUC cai 0,017. O topo da importância passa a ser solidão, idade, suporte social, limitação de trabalho por saúde e dor crônica.
- **Sem o bloco e sem a solidão:** o ROC-AUC é 0,811 e a precisão 45,4%, **2,1 vezes a prevalência** (21,6%). O modelo continua bem acima do acaso e o recall se mantém em 72%. As variáveis mais importantes passam a ser idade, suporte social, limitação de trabalho por saúde, autoavaliação de saúde e dor crônica, ou seja, **saúde física e funcionamento**.
- **Mesmo padrão dos outros modelos:** a queda total vai de 0,856 para 0,811 (0,045), contra 0,841 para 0,804 no Random Forest e 0,816 para 0,778 no KNN.
- **Limite do teste:** ele retira só as variáveis que identificamos como suspeitas. Outras, como suporte social, também podem se relacionar com sintomas. O teste mostra robustez, não ausência total de sobreposição.

# 6. Priorizando o recall: rastreio em saúde

Em um rastreio de depressão, os dois erros não custam o mesmo. Um **falso negativo** é uma pessoa com depressão que o modelo não identifica e que, portanto, não é encaminhada para avaliação. Um **falso positivo** é uma pessoa sem depressão que passa por uma avaliação a mais. Por isso é preferível encontrar mais casos, mesmo que isso piore os falsos positivos. Esta seção mostra quanto recall dá para ganhar **baixando o limiar** de decisão do XGBoost (até aqui, probabilidade 0,5) e quanto isso custa: o modelo sinaliza mais pessoas, o recall sobe e a precisão cai.

**Como o limiar é escolhido, sem olhar o teste:** calculamos a probabilidade prevista dos registros de treino com validação cruzada de 5 partes (`cross_val_predict`) e, para cada **recall-alvo** (80%, 85%, 90% e 95%), usamos o limiar de probabilidade que alcança esse recall entre os casos de depressão do treino. Depois aplicamos o limiar ao teste e medimos o que realmente aconteceu. Por isso o recall obtido no teste pode ficar um pouco acima ou abaixo do alvo.

**Como ler a tabela:** `Recall` e `Precisão` são ponderados pelo `PESO_AMOSTRAL`, como nas seções anteriores. `TP`, `FN`, `FP` e `TN` são **contagens de registros** do teste, sem peso, e `% da amostra sinalizada` e `Sinalizados por caso encontrado` também usam contagens. Este último é quantas pessoas precisam ser avaliadas para achar um caso de depressão.

In [ ]:
ALVOS_RECALL = [0.80, 0.85, 0.90, 0.95]


def analisar_limiares(colunas_remover, **parametros):
    """Escolhe o limiar no treino (validação cruzada) para cada recall-alvo e mede o resultado no teste."""
    _, _, X_tr_p, X_te_p = preparar_matrizes(colunas_remover)
    proba_te = novo_xgb(**parametros).fit(X_tr_p, y_tr).predict_proba(X_te_p)[:, 1]
    # O limiar de cada recall-alvo vem das probabilidades de validação cruzada do TREINO; o teste só mede o recall obtido
    proba_cv = cross_val_predict(novo_xgb(**parametros), X_tr_p, y_tr, cv=5, method='predict_proba')[:, 1]
    proba_positivos = proba_cv[(y_tr == 1).to_numpy()]

    linhas, matrizes = {}, {}
    for alvo in [None] + ALVOS_RECALL:
        limiar = 0.5 if alvo is None else np.quantile(proba_positivos, 1 - alvo)
        nome = 'Padrão (limiar 0,5)' if alvo is None else f'Recall-alvo {alvo:.0%}'
        metricas, matrizes[nome] = avaliar(proba_te, limiar)
        tn, fp, fn, tp = matrizes[nome].ravel()
        linhas[nome] = {
            'Limiar': limiar,
            'Recall': metricas['Recall (depressão)'],
            'Precisão': metricas['Precisão (depressão)'],
            'TP': tp, 'FN': fn, 'FP': fp, 'TN': tn,
            '% da amostra sinalizada': (tp + fp) / len(y_te),
            'Sinalizados por caso encontrado': (tp + fp) / tp,
        }
    return pd.DataFrame.from_dict(linhas, orient='index'), matrizes

In [ ]:
tabela_lim_b, matrizes_lim_b = analisar_limiares(COLUNAS_B, **busca.best_params_)

print(f'Cenário B (modelo ajustado). Prevalência de depressão no teste (ponderada): {prevalencia:.1%}')
print('\nMatriz de confusão com recall-alvo de 90% (nº de registros do teste; linhas = real, colunas = previsto):')
print(matrizes_lim_b['Recall-alvo 90%'])
tabela_lim_b.round(3)

### Resultado (cenário B)

- **Mais recall, mais falsos positivos.** Do limiar padrão (0,5) para o recall-alvo de 90%, o recall vai de 73,2% para **88,4%** e os casos perdidos caem de 243 para **98**. O custo são os falsos positivos, que dobram (de 690 para **1.393**), e a precisão, que cai de 52,1% para 38,8%.
- **Custo em avaliações.** Para cada caso encontrado, são avaliadas 2,0 pessoas no limiar padrão, 2,7 no recall-alvo de 90% e 3,1 no de 95%.
- **Continua muito melhor que o acaso.** No recall-alvo de 90%, o modelo sinaliza 50,5% da amostra e encontra 837 dos 935 casos. Um sorteio que sinalizasse a mesma fração encontraria cerca de 470. A precisão de 38,8% é 1,8 vez a prevalência (21,6%), e mesmo no alvo de 95% ela é 1,5 vez (33,1%).
- **O recall obtido no teste fica perto do alvo.** Ele ficou entre 0,8 e 2,8 pontos abaixo dos alvos (77,2%, 84,2%, 88,4% e 93,4%), porque o limiar vem do treino.
- **Qual alvo usar é uma decisão de saúde, não estatística.** Depende do custo de avaliar cada pessoa sinalizada e da capacidade do serviço. Se a prioridade é não deixar casos para trás, o alvo de 90% é um ponto de partida razoável: cerca de 4 em cada 10 pessoas sinalizadas têm depressão, e 98 dos 935 casos (10%) ficam sem identificação.

### Mesmo teste no cenário mais conservador

Repetimos a análise sem o bloco cognitivo e sem a solidão (seção 5), as variáveis mais próximas de sintomas depressivos.

In [ ]:
tabela_lim_cons, _ = analisar_limiares(COLUNAS_B + BLOCO_COGNITIVO + SOLIDAO, **busca.best_params_)
tabela_lim_cons.round(3)

- No limiar padrão, o cenário conservador encontra 676 dos 935 casos (recall de 71,7%, precisão de 45,4%). Para chegar a um recall de **90,0%** no teste, o modelo sinaliza 59,4% da amostra, com 1.785 falsos positivos e precisão de **33,0%** (1,5 vez a prevalência).
- Portanto, **dá para atingir um recall alto mesmo sem as variáveis cognitivas e de solidão**, mas o custo é maior: 3,1 pessoas avaliadas por caso encontrado, contra 2,7 no cenário B completo.

# 7. O que mais dá para fazer pelo recall no XGBoost

Há dois tipos de alavanca para encontrar mais casos. **Deslocar o ponto de operação** (baixar o limiar, como na seção 6, ou dar mais peso à classe com depressão) aumenta o recall, mas também os falsos positivos, ao longo da mesma curva de precisão × recall. **Melhorar o modelo** sobe a curva: mais recall com os mesmos falsos positivos.

Para separar os dois, usamos duas medidas que **não dependem do limiar**: o **PR-AUC** e a **precisão com recall de 90%**, que é a maior precisão que o modelo alcança quando exigimos que ele encontre pelo menos 90% dos casos do teste. Se uma variante aumenta o recall no limiar padrão, mas deixa essas duas medidas iguais, ela só moveu o ponto de operação. Um modelo sem informação teria PR-AUC e precisão iguais à prevalência (21,6%).

Variantes testadas no cenário B, com os hiperparâmetros ajustados:

1. **`scale_pos_weight` maior que o balanceado** (negativos ÷ positivos, cerca de 3,7): 1,5, 2 e 3 vezes esse valor.
2. **Sem `scale_pos_weight`**, para ver o efeito do peso sobre o ranking do modelo.
3. **Busca de hiperparâmetros pelo PR-AUC** em vez do ROC-AUC, para ver se um critério mais próximo do recall escolhe outro modelo.

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve


def precisao_no_recall(escore, recall_alvo=0.90):
    """Maior precisão (ponderada) que o modelo alcança no teste exigindo ao menos `recall_alvo` de recall.

    ATENÇÃO: o limiar é escolhido olhando o próprio teste (cenário ideal). Serve só para comparar modelos e é um
    limite superior otimista; o resultado realista é o da tabela de limiares escolhidos na validação cruzada."""
    precisao, recall, _ = precision_recall_curve(y_te, escore, sample_weight=w_te)
    return precisao[recall >= recall_alvo].max()


def resumir(escore, limiar=0.5):
    """Medidas que não dependem do limiar e o resultado no limiar padrão."""
    metricas, matriz = avaliar(escore, limiar)
    tn, fp, fn, tp = matriz.ravel()
    return {
        'ROC-AUC': metricas['ROC-AUC'],
        'PR-AUC': average_precision_score(y_te, escore, sample_weight=w_te),
        'Precisão com recall de 90%': precisao_no_recall(escore),
        'Recall': metricas['Recall (depressão)'],
        'Precisão': metricas['Precisão (depressão)'],
        'Casos perdidos (FN)': fn,
        'Falsos positivos (FP)': fp,
    }

In [ ]:
VARIANTES_PESO = {
    'Base (scale_pos_weight balanceado)': 1,
    'scale_pos_weight × 1,5': 1.5,
    'scale_pos_weight × 2': 2,
    'scale_pos_weight × 3': 3,
    'Sem scale_pos_weight': 1 / SCALE_POS,
}
escores = {}
for nome, multiplo in VARIANTES_PESO.items():
    modelo = novo_xgb(**{**busca.best_params_, 'scale_pos_weight': SCALE_POS * multiplo}).fit(X_tr_b, y_tr)
    escores[nome] = modelo.predict_proba(X_te_b)[:, 1]

tabela_alavancas = pd.DataFrame.from_dict({nome: resumir(e) for nome, e in escores.items()}, orient='index')
print(f'Referência de acaso: PR-AUC e precisão com recall de 90% iguais à prevalência ({prevalencia:.1%})\n')
tabela_alavancas.round(4)

In [ ]:
# Mesma busca da seção 4, mas otimizando o PR-AUC (average_precision)
busca_ap = RandomizedSearchCV(
    novo_xgb(), ESPACO_BUSCA, n_iter=20, scoring='average_precision', cv=3,
    random_state=RANDOM_STATE, n_jobs=1,
)
busca_ap.fit(X_tr_b, y_tr)
print('Melhores hiperparâmetros pelo PR-AUC:', busca_ap.best_params_)
print('Iguais aos da busca pelo ROC-AUC?', busca_ap.best_params_ == busca.best_params_)

### Resultado

- **Aumentar o `scale_pos_weight` eleva o recall, mas só move o ponto de operação.** Com 1,5, 2 e 3 vezes o valor balanceado, o recall no limiar padrão vai de 73,2% para 81,5%, 85,5% e 89,7%, e os casos perdidos caem de 243 para 169, 129 e 85. O custo são os falsos positivos, que sobem de 690 para 982, 1.185 e 1.515. O ROC-AUC (0,8545 a 0,8566), o PR-AUC (0,670 a 0,675) e a precisão com recall de 90% (0,362 a 0,369) ficam na mesma faixa em todas as variantes. O custo em falsos positivos é o mesmo de baixar o limiar (seção 6).
- **Sem `scale_pos_weight`, o ranking do modelo é o mesmo.** O ROC-AUC é 0,857 e a precisão com recall de 90% é 0,369, mas o recall cai para 43,3% (512 casos perdidos). O peso não melhora o que o modelo aprende, só muda onde ele opera por padrão.
- **Buscar os hiperparâmetros pelo PR-AUC não muda nada.** A busca escolheu exatamente os mesmos hiperparâmetros da seção 4, que foram escolhidos pelo ROC-AUC. Um critério mais próximo do recall não encontra outro modelo melhor.
- **Conclusão para o XGBoost:** para encontrar mais casos, o caminho é escolher o ponto de operação (o limiar da seção 6 ou um `scale_pos_weight` maior). Nenhuma variante testada sobe a curva, então não há recall "de graça": cada caso a mais custa falsos positivos. O limite está na informação das variáveis, não nos hiperparâmetros.

# 8. Interpretação e conclusão

**ROC-AUC (não depende do limiar):**

| Modelo | ROC-AUC (A) | ROC-AUC (B) | B sem bloco cognitivo e sem solidão |
|---|---|---|---|
| XGBoost ajustado | 0,915 (base) | 0,856 | 0,811 |
| SVM linear ajustado | 0,915 (base) | 0,856 | 0,813 |
| KNN com colunas ponderadas | - | 0,843 | 0,796 |
| Random Forest | 0,915 | 0,841 | 0,804 |
| KNN ajustado (seção 4) | 0,833 (base, `k = 15`) | 0,816 | 0,778 |

**Pontos de operação no cenário B** (modelos ajustados). Recall e precisão são ponderados pelo `PESO_AMOSTRAL` e os falsos positivos são contagens do teste. Cada modelo tem um limiar padrão diferente. No recall-alvo de 90%, o limiar vem do treino (seção 6 deste notebook e seções equivalentes dos outros):

| Modelo | Limiar padrão: recall e precisão | Recall-alvo de 90%: recall obtido | Precisão | Falsos positivos | % da amostra sinalizada |
|---|---|---|---|---|---|
| SVM linear | 72,4% e 52,7% | 86,9% | 38,5% | 1.341 | 48,9% |
| XGBoost | 73,2% e 52,1% | 88,4% | 38,8% | 1.393 | 50,5% |
| Random Forest | 59,7% e 57,7% | 88,3% | 38,0% | 1.445 | 51,7% |
| KNN com colunas ponderadas | 66,8% e 52,9% | 88,0% | 37,0% | 1.530 | 53,7% |
| KNN ajustado (seção 4) | 45,6% e 60,8% | 87,2% | 34,0% | 1.679 | 56,8% |

**O que mais melhora o recall em cada algoritmo** (seção 7 deste notebook e seções equivalentes dos outros):

| Algoritmo | Alavanca testada | Recall no limiar padrão | Melhora a curva de precisão × recall? |
|---|---|---|---|
| Random Forest | peso 6 para a classe positiva | 59,7% → 71,5% | Não: só desloca o ponto de operação |
| KNN | colunas ponderadas pelo SVM L1 | 45,6% → 66,8% | **Sim:** PR-AUC 0,600 → 0,646 e ROC-AUC 0,816 → 0,843 |
| SVM linear | peso 5 e 8 para a classe positiva | 72,4% → 78,8% e 84,7% | Não |
| XGBoost | `scale_pos_weight` × 2 | 73,2% → 85,5% | Não |

- **A hipótese se sustenta com o XGBoost.** Sem nenhum sintoma de saúde mental, o modelo fica 0,36 de ROC-AUC acima do acaso (0,856) e encontra cerca de 73% dos casos de depressão, com precisão de 52% contra 21,6% de prevalência. Dados físicos, funcionais e de contexto social carregam informação real sobre depressão.
- **Como esperado, é pior que com ansiedade.** A perda é de cerca de 0,06 de ROC-AUC e de 8 pontos de recall. No cenário A, 7 das 10 variáveis mais importantes são de ansiedade.
- **O resultado é robusto.** Retirar as variáveis mais próximas de sintomas depressivos (cognição e solidão) reduz o ROC-AUC em 0,045, e o modelo continua bem acima do acaso. Sem elas, o sinal vem de idade, suporte social, limitação de trabalho por saúde, autoavaliação de saúde e dor crônica.
- **O XGBoost empata com o SVM linear e fica acima do Random Forest e do KNN** em ROC-AUC no cenário B (0,856 contra 0,856, 0,841 e 0,816, ou 0,843 para o KNN com colunas ponderadas). No teste de sensibilidade o quadro é o mesmo (XGBoost 0,811, SVM 0,813, Random Forest 0,804 e KNN ponderado 0,796). A diferença para o Random Forest é pequena (0,015) e vem de um único split. O empate com um modelo linear sugere que o sinal das variáveis é em grande parte **aditivo**.
- **Para o rastreio, o ponto de operação pesa mais que o modelo.** Com o recall-alvo de 90%, o XGBoost sinaliza 50,5% da amostra e encontra 837 dos 935 casos, deixando 98 sem identificação (243 no limiar padrão). O custo são 1.393 falsos positivos (690 no limiar padrão) e uma precisão de 38,8%, ainda 1,8 vez a prevalência. Mesmo no cenário conservador (sem cognição e sem solidão), um recall de 90% é alcançável, com precisão de 33,0%.
- **No mesmo recall, SVM, XGBoost e Random Forest se aproximam.** Com o alvo de 90%, os três sinalizam de 49% a 52% da amostra e têm precisão de 38% a 39%, uma diferença dentro da variação de um único teste. O recall mais baixo do Random Forest no limiar padrão (59,7%) vem do limiar de 0,5, não de um pior ordenamento do risco. Os dois KNN precisam de mais falsos positivos para o mesmo recall (1.530 e 1.679, contra 1.341 a 1.445).
- **Nenhuma variante do XGBoost sobe a curva.** Um `scale_pos_weight` 1,5, 2 e 3 vezes maior leva o recall no limiar padrão a 81,5%, 85,5% e 89,7%, mas só desloca o ponto de operação, e a busca pelo PR-AUC escolhe os mesmos hiperparâmetros. Entre os quatro algoritmos, só no KNN mudar o modelo ajudou: ponderar as colunas pelos coeficientes de um SVM L1 subiu o PR-AUC de 0,600 para 0,646.

## Limitações

- O `PESO_AMOSTRAL` entra só nas métricas, não no treino. O desbalanceamento é tratado com `scale_pos_weight` e limiar de 0,5.
- O recall-alvo (80% a 95%) é uma decisão de saúde, não estatística. O notebook mostra o custo de cada alvo, mas não define qual usar. O limiar é escolhido no treino, e o recall obtido no teste varia alguns pontos em torno do alvo.
- Um único split treino/teste. A busca de hiperparâmetros usa validação cruzada, mas a comparação final usa um só teste, com cerca de 900 casos positivos. Diferenças de um a dois pontos entre variantes ficam dentro dessa variação.
- Registros completos apenas (22.084 de 24.215).
- Os modelos mostram **associação preditiva**, não causalidade.